# Phase 1 — BERT Fine-Tuning (Baseline Classifier)
**DiagGen+ | Advanced AI Group Project 2026**

## Objectives
1. Load the pre-split imbalanced dataset from `data/processed/`.
2. Fine-tune BioBERT on the imbalanced training set.
3. Evaluate on the held-out test set — record baseline metrics.
4. Generate confusion matrix and per-class F1 chart.
5. Save the trained model to `models/bert/`.

## Why imbalanced training data?
We deliberately train on the imbalanced dataset (rare classes at 20 samples each) to **demonstrate the problem** 
that Phase 2 (VAE augmentation) will solve. The baseline model will show poor F1 on rare disease classes. 
Phase 2 will retrain on the augmented dataset and show measurable improvement.

> **Gate 1 complete** when: model trained, baseline metrics recorded, Streamlit app returns real predictions.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.config_loader import cfg, get
from src.utils.logger import get_logger
from src.preprocessing.cleaner import preprocess_batch
from src.models.classifier import DiagnosisClassifier
from src.training.trainer import load_processed_splits, build_label_maps
from src.evaluation.metrics import evaluate, plot_confusion_matrix, plot_class_f1_comparison

logger = get_logger('phase1_bert')
FIGURES_DIR  = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = Path('../data/processed')
print('Imports OK')

Imports OK


## 1. Load Dataset Splits

In [2]:
train_df, val_df, test_df = load_processed_splits()

# Load rare class list (produced by Phase 0)
rare_meta    = json.loads((PROCESSED_DIR / 'rare_classes.json').read_text())
RARE_DISEASES = rare_meta['rare_diseases']
print(f'Rare classes ({len(RARE_DISEASES)}): {RARE_DISEASES}')

# Quick sanity check on class distribution in training set
print('\nTraining set class counts:')
print(train_df['disease'].value_counts().to_string())

2026-06-13 22:59:59 | INFO     | src.training.trainer | Loaded splits — Train: 712 | Val: 89 | Test: 89
Rare classes (6): ['dengue', 'drug reaction', 'impetigo', 'malaria', 'psoriasis', 'typhoid']

Training set class counts:
disease
allergy                            40
diabetes                           40
varicose veins                     40
bronchial asthma                   40
arthritis                          40
chicken pox                        40
cervical spondylosis               40
hypertension                       40
gastroesophageal reflux disease    39
common cold                        39
peptic ulcer disease               38
pneumonia                          38
fungal infection                   38
urinary tract infection            38
migraine                           34
jaundice                           32
dengue                             16
impetigo                           16
drug reaction                      16
malaria                            16
typhoid

## 2. Build Label Maps & Preprocess Text

In [3]:
# Build consistent label maps from full combined set
all_df = pd.concat([train_df, val_df, test_df])
label2id, id2label, le = build_label_maps(all_df)

print(f'Number of classes: {len(label2id)}')
print(f'Label map: {label2id}')

2026-06-13 23:00:34 | INFO     | src.training.trainer | Label map built: 22 classes.
Number of classes: 22
Label map: {'allergy': 0, 'arthritis': 1, 'bronchial asthma': 2, 'cervical spondylosis': 3, 'chicken pox': 4, 'common cold': 5, 'dengue': 6, 'diabetes': 7, 'drug reaction': 8, 'fungal infection': 9, 'gastroesophageal reflux disease': 10, 'hypertension': 11, 'impetigo': 12, 'jaundice': 13, 'malaria': 14, 'migraine': 15, 'peptic ulcer disease': 16, 'pneumonia': 17, 'psoriasis': 18, 'typhoid': 19, 'urinary tract infection': 20, 'varicose veins': 21}


In [4]:
# Preprocess symptom text for all splits
# Uses the column 'symptoms_clean' if already preprocessed in Phase 0,
# otherwise falls back to running preprocessing now.
def get_texts(df):
    if 'symptoms_clean' in df.columns:
        return df['symptoms_clean'].fillna('').tolist()
    print('symptoms_clean not found — running preprocessing now...')
    return preprocess_batch(df['symptoms'].tolist())

print('Preparing text...')
train_texts  = get_texts(train_df)
val_texts    = get_texts(val_df)
test_texts   = get_texts(test_df)

train_labels = le.transform(train_df['disease'].tolist()).tolist()
val_labels   = le.transform(val_df['disease'].tolist()).tolist()
test_labels  = le.transform(test_df['disease'].tolist()).tolist()

print(f'Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}')

Preparing text...
symptoms_clean not found — running preprocessing now...
symptoms_clean not found — running preprocessing now...
symptoms_clean not found — running preprocessing now...
Train: 712 | Val: 89 | Test: 89


## 3. Fine-Tune BioBERT

This cell runs the full fine-tuning loop. **Expected runtime: 15–40 minutes on CPU; 5–10 minutes on GPU.**
If this is too slow, swap `bert.base_model` in `config/config.yaml` to `'bert-base-uncased'`.

In [ ]:
clf = DiagnosisClassifier(label2id, id2label)

print(f'Base model:   {get("bert.base_model")}')
print(f'Max length:   {get("bert.max_length")}')
print(f'Epochs:       {get("bert.training.num_epochs")}')
print(f'Batch size:   {get("bert.training.batch_size")}')
print(f'Learning rate:{get("bert.training.learning_rate")}')
print()

train_ds = clf.prepare_dataset(train_texts, train_labels)
val_ds   = clf.prepare_dataset(val_texts,   val_labels)

clf.train(train_ds, val_ds)
print('\n✅ Training complete. Model saved to models/bert/')

2026-06-13 23:02:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-13 23:02:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dmis-lab/biobert-base-cased-v1.2/67c9c25b46986521ca33df05d8540da1210b3256/config.json "HTTP/1.1 200 OK"
2026-06-13 23:02:39 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/dmis-lab/biobert-base-cased-v1.2/67c9c25b46986521ca33df05d8540da1210b3256/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

c:\Users\tharu\.conda\envs\diaggen-plus\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tharu\.cache\huggingface\hub\models--dmis-lab--biobert-base-cased-v1.2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


2026-06-13 23:02:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-13 23:02:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"


2026-06-13 23:02:40 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-13 23:02:40 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/dmis-lab/biobert-base-cased-v1.2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-06-13 23:02:40 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/dmis-lab/biobert-base-cased-v1.2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-06-13 23:02:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/vocab.txt "HTTP/1.1 307 Temporary Redirect"
2026-06-13 23:02:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dmis-lab/biobert-base-cased-v1.2/67c9c25b46986521ca33df05d8540da1210b3256/vocab.txt "HTTP/1.1 200 OK"
2026-06-13 23:02:4

...

Map:   0%|          | 0/89 [00:00<?, ? examples/s]

2026-06-13 23:02:43 | INFO     | src.models.classifier | Loading base model: dmis-lab/biobert-base-cased-v1.2
2026-06-13 23:02:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-13 23:02:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dmis-lab/biobert-base-cased-v1.2/67c9c25b46986521ca33df05d8540da1210b3256/config.json "HTTP/1.1 200 OK"


[transformers] You passed `num_labels=22` which is incompatible to the `id2label` map of length `2`.


2026-06-13 23:02:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-06-13 23:02:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-06-13 23:02:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

## 4. Baseline Evaluation

In [ ]:
# Generate predictions on the test set
from tqdm import tqdm

print('Running inference on test set...')
y_pred = []
for text in tqdm(test_texts):
    result = clf.predict(text)
    y_pred.append(label2id[result['top_disease']])

y_true = test_labels
label_names = [id2label[i] for i in range(len(id2label))]

baseline_metrics = evaluate(y_true, y_pred, label_names)
print(f"\nBaseline Accuracy:  {baseline_metrics['accuracy']:.4f}")
print(f"Baseline Macro F1:  {baseline_metrics['macro_f1']:.4f}")

In [ ]:
# ── Per-class F1 — highlight rare classes ────────────────────────────────────
report_df = pd.DataFrame(baseline_metrics['report']).T
report_df = report_df.loc[label_names]   # keep only disease rows
report_df['is_rare'] = report_df.index.isin(RARE_DISEASES)

print('Per-class F1 scores:')
print(report_df[['precision','recall','f1-score','support','is_rare']].to_string())

print(f"\nMean F1 — common classes: {report_df[~report_df['is_rare']]['f1-score'].mean():.4f}")
print(f"Mean F1 — rare classes:   {report_df[report_df['is_rare']]['f1-score'].mean():.4f}")
print('(Rare class F1 should be noticeably lower — this is the problem Phase 2 solves)')

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
plot_confusion_matrix(
    y_true, y_pred, label_names,
    save_path=FIGURES_DIR / 'baseline_confusion_matrix.png'
)

In [ ]:
# ── Per-class F1 bar chart — colour rare classes red ─────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
f1_scores = [report_df.loc[n, 'f1-score'] if n in report_df.index else 0 for n in label_names]
colours   = ['#E74C3C' if n in RARE_DISEASES else '#2E75B6' for n in label_names]

bars = ax.bar(label_names, f1_scores, color=colours)
ax.axhline(y=0.80, color='green', linestyle='--', alpha=0.7, label='0.80 target F1')
ax.set_ylabel('F1 Score')
ax.set_title('Baseline BERT — Per-Class F1 (Red = simulated rare classes)')
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'baseline_per_class_f1.png', dpi=150)
plt.show()
print('Chart saved.')

In [ ]:
# ── Save baseline metrics for comparison in Phase 2 ──────────────────────────
import json
(PROCESSED_DIR / 'baseline_metrics.json').write_text(
    json.dumps({
        'accuracy':    baseline_metrics['accuracy'],
        'macro_f1':    baseline_metrics['macro_f1'],
        'weighted_f1': baseline_metrics['weighted_f1'],
        'per_class_f1': {
            n: report_df.loc[n, 'f1-score']
            for n in label_names if n in report_df.index
        },
        'rare_classes_mean_f1': float(report_df[report_df['is_rare']]['f1-score'].mean()),
        'common_classes_mean_f1': float(report_df[~report_df['is_rare']]['f1-score'].mean()),
    }, indent=2)
)
print('Baseline metrics saved to data/processed/baseline_metrics.json')

## 5. Smoke Test — Live Prediction

In [ ]:
# Quick sanity check with a few manual inputs
from src.preprocessing.cleaner import preprocess

test_cases = [
    'I have had a high fever, severe headache, and pain behind my eyes for 3 days',
    'I feel very tired, my skin is itchy, and my urine is dark yellow',
    'I have joint pain, skin rash, and hair loss',
]

for symptom_text in test_cases:
    clean   = preprocess(symptom_text)
    result  = clf.predict(clean)
    print(f'Input:    {symptom_text[:70]}')
    print(f'Top pred: {result["top_disease"]} ({result["top_confidence"]:.1%})')
    print(f'Above threshold ({get("bert.confidence_threshold"):.0%}): {result["above_threshold"]}')
    for p in result['predictions']:
        print(f'          {p["disease"]:<35} {p["confidence"]:.1%}')
    print()

## 6. Wire the Classifier into the Streamlit App

Now that the model is trained, update `app/pages/diagnosis.py` to replace the placeholder with real predictions.
The cell below prints the exact code block to paste in.

In [ ]:
print('''Replace the _process() function in app/pages/diagnosis.py with:

from src.models.classifier import DiagnosisClassifier
from src.preprocessing.cleaner import preprocess
from src.qa.llm_client import call, format_prompt
from src.utils.config_loader import get
import json, pathlib

@st.cache_resource
def _load_classifier():
    meta     = json.loads(pathlib.Path("data/processed/rare_classes.json").read_text())
    all_data = pd.read_csv("data/processed/train_imbalanced.csv")
    from src.training.trainer import build_label_maps
    import pandas as pd
    label2id, id2label, _ = build_label_maps(all_data)
    clf = DiagnosisClassifier(label2id, id2label)
    return clf.load()

def _process(user_input: str) -> str:
    clf    = _load_classifier()
    clean  = preprocess(user_input)
    result = clf.predict(clean)
    preds  = result["predictions"]
    preds_str = "\\n".join(
        f"{i+1}. {p[\'disease\']} — {p[\'confidence\']:.1%}"
        for i, p in enumerate(preds)
    )
    prompt   = format_prompt("explanation.template",
                             symptoms=user_input, predictions=preds_str,
                             top_disease=result["top_disease"])
    explanation = call(prompt)
    return f"**Top prediction:** {result[\'top_disease\']} ({result[\'top_confidence\']:.1%})\\n\\n{explanation}"
''')

## Phase 1 Summary ✅ — Gate 1 Reached

Fill in after running:

| Metric | Value |
|--------|-------|
| Baseline Accuracy | — |
| Baseline Macro F1 | — |
| Common class mean F1 | — |
| Rare class mean F1 | — |
| Model checkpoint | `models/bert/` |

**The gap between common class F1 and rare class F1 is the core problem statement for Phase 2.**

> Copy this table into the report Results section as 'Table 1: Baseline Model Performance'.